# Libraries

In [32]:
#Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import os
import csv
from datetime import datetime

# Global variables

In [33]:
#Definindo o diretório onde os arquivos CSVs estão localizados e o nome do arquivo de saída
DATASET_DIR = "Dataset"
OUTPUT_FILE = "Schema/schema_banvic.sql"

# Global functions

In [34]:
#Função para mostrar as colunas dos arquivos CSVs em uma pasta especifica
def show_columns(folder):
    """ Lê todos os arquivos CSVs de uma pasta e apresenta as colunas existentes em cada arquivo. """

    files = sorted(
        file for file in os.listdir(folder)
        if file.lower().endswith(".csv")
    )

    if not files:
        print(f"Nenhum arquivo CSV encontrado na pasta '{folder}'.")
        return

    print(f"Arquivos encontrados: {len(files)}\n")

    for file in files:
        path = os.path.join(folder, file)

        df = pd.read_csv(path)

        print("=" * 60)
        print(f"Arquivo: {file}")
        print(f"Quantidade de colunas: {len(df.columns)}")
        print("-" * 60)
        
        for col in df.columns:
            print(f"- {col}")

        print()

In [35]:
#Função para identificar colunas que são identificadores (id), cpfs/cnpjs, etc., que devem ser tratados como TEXT
def id_type_column(column_name):
    """ Verifica se a coluna representa um identificador (id), cpfs/cnpjs, etc. para ser tratada separadamente. """

    #Informações obtidas da função show_columns, que mostra as colunas dos arquivos CSV na pasta "Dataset"
    column_name = column_name.strip().lower()
    return (
        column_name == "id"
        or column_name.startswith("cod_")
        or column_name.startswith("num_")
        or column_name == "cpfcnpj"
        or column_name == "cep"
        or column_name == "cpf"
        or column_name == "cnpj"
    )

#Função para inferir o tipo de dados de cada coluna com base nos valores encontrados nos CSVs
def infer_type(column_name, values):
    """ Infere o tipo de SQL de acordo com o que o PostgreSQL aceita no schema com base nos valores encontrados. """
    
    values = [value.strip() for value in values if value.strip()]

    #Colunas de IDs serão sempre tratados como TEXT
    if id_type_column(column_name):
        return "TEXT"

    values = [
        value.strip()
        for value in values
        if value.strip() != ""
    ]

    #TEXT
    if not values:
        return "TEXT"

    #INTEGER
    try:
        for value in values:
            int(value)
        return "INTEGER"
    except ValueError:
        pass

    #NUMERIC
    try:
        for value in values:
            float(value)
        return "NUMERIC"
    except ValueError:
        pass

    #DATE/TIMESTAMP
    date_formats = [
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S"
    ]
    for date_format in date_formats:
        try:
            for value in values:
                datetime.strptime(value, date_format)
            if "H" in date_format:
                return "TIMESTAMP"
            return "DATE"
        except ValueError:
            continue
    return "TEXT"

In [36]:
#Funções para tratar nomes de tabelas e colunas para o PostgreSQL
def created_table_name(filename):
    """ Utiliza o nome do arquivo CSV como nome da tabela SQL."""
    
    table_name = os.path.splitext(filename)[0]

    return table_name.lower()

def created_column_name(column):
    """ Normaliza o nome das colunas para PostgreSQL."""

    #tratamento de possiveis espaços e caracteres especiais no nome da coluna
    column = column.strip()
    column = column.replace(" ", "_")
    column = column.replace("-", "_")

    return column.lower()

In [37]:
#Função principal para gerar o schema SQL a partir dos arquivos CSVs
def generated_schema():
    """ Gera o schema SQL a partir dos arquivos CSVs no diretório especificado. """

    sql_statements = []
    files = sorted(os.listdir(DATASET_DIR))

    for filename in files:
        #Apenas selecionar arquivos .csv no diretório especificado
        if not filename.lower().endswith(".csv"):
            continue

        filepath = os.path.join(DATASET_DIR, filename)
        table_name = created_table_name(filename)
    
        print(f"Processando: {filename}")

        with open(filepath, "r", encoding="utf-8-sig", newline="") as csvfile:

            reader = csv.reader(csvfile)
            header = next(reader)
            rows = list(reader)
            columns = list(zip(*rows)) if rows else [[] for _ in header]
            column_definitions = []

            for column_name, values in zip(header, columns):
                original_column_name = column_name #Para verifcar as colunas Ids
                column_name = created_column_name(column_name)
                data_type = infer_type(original_column_name, values)
                column_definitions.append(
                    f'    "{column_name}" {data_type}'
                )

            create_table = (
                f'CREATE TABLE "{table_name}" (\n'
                + ",\n".join(column_definitions)
                + "\n);\n"
            )

            sql_statements.append(create_table)

    with open(OUTPUT_FILE, "w", encoding="utf-8") as sqlfile:
        sqlfile.write(
            "-- Schema gerado automaticamente\n\n"
        )
        sqlfile.write(
            "\n".join(sql_statements)
        )

# KPIs + business questions

In [38]:
#Chamando a função para mostrar as colunas dos arquivos CSV na pasta "Dataset"
show_columns(DATASET_DIR)

Arquivos encontrados: 7

Arquivo: agencias.csv
Quantidade de colunas: 7
------------------------------------------------------------
- cod_agencia
- nome
- endereco
- cidade
- uf
- data_abertura
- tipo_agencia

Arquivo: clientes.csv
Quantidade de colunas: 10
------------------------------------------------------------
- cod_cliente
- primeiro_nome
- ultimo_nome
- email
- tipo_cliente
- data_inclusao
- cpfcnpj
- data_nascimento
- endereco
- cep

Arquivo: colaborador_agencia.csv
Quantidade de colunas: 2
------------------------------------------------------------
- cod_colaborador
- cod_agencia

Arquivo: colaboradores.csv
Quantidade de colunas: 8
------------------------------------------------------------
- cod_colaborador
- primeiro_nome
- ultimo_nome
- email
- cpf
- data_nascimento
- endereco
- cep

Arquivo: contas.csv
Quantidade de colunas: 9
------------------------------------------------------------
- num_conta
- cod_cliente
- cod_agencia
- cod_colaborador
- tipo_conta
- data_aber

In [39]:
#Chamando função principal para gerar o schema SQL a partir dos arquivos CSVs
generated_schema()

Processando: agencias.csv
Processando: clientes.csv
Processando: colaborador_agencia.csv
Processando: colaboradores.csv
Processando: contas.csv
Processando: propostas_credito.csv
Processando: transacoes.csv


# EDA + insights